In [1]:
import pandas as pd
import json
from datasets import load_dataset

In [2]:
ds1 = load_dataset("AnishJoshi/nl2bash-custom", split="train")
df1 = ds1.to_pandas()
  
ds2 = load_dataset("mecha-org/linux-command-dataset", split="train")
df2 = ds2.to_pandas()

ds3 = load_dataset("harpomaxx/unix-commands", split="train")
df3 = ds3.to_pandas()

print(f"nl2bash-custom loaded: {len(df1)} rows | columns: {list(df1.columns)}")
print(f"linux-command-dataset loaded: {len(df2)} rows | columns: {list(df2.columns)}")
print(f"unix-commands loaded: {len(df3)} rows | columns: {list(df3.columns)}")

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/8669 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

unix-commands-dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

nl2bash-custom loaded: 19658 rows | columns: ['bash_code', 'nl_command', 'srno']
linux-command-dataset loaded: 8669 rows | columns: ['input', 'output']
unix-commands loaded: 100 rows | columns: ['input', 'instruction', 'output']


In [3]:
df1 = df1.rename(columns={
    "nl_command": "instruction",
    "bash_code":  "output"
})
 
df2 = df2.rename(columns={
    "input": "instruction"
})
 
df3 = df3.rename(columns={
    "instruction": "Prompt",
    "input": "instruction"
})
 
print("All fields normalized to: instruction, output")

All fields normalized to: instruction, output


In [4]:
df1 = df1[["instruction", "output"]]
df2 = df2[["instruction", "output"]]
df3 = df3[["instruction", "output"]]

In [8]:
def clean_dataframe(df, name):
    original_count = len(df)

    df["instruction"] = df["instruction"].astype(str).str.strip()
    df["output"]      = df["output"].astype(str).str.strip()

    df = df[(df["instruction"].notna()) & (df["instruction"].str.len() > 0)]
    df = df[(df["output"].notna())      & (df["output"].str.len() > 0)]

    df = df[df["output"].str.len() >= 3]

    df = df.drop_duplicates(subset=["instruction"])

    cleaned_count = len(df)
    dropped = original_count - cleaned_count
    print(f"  {name}: {original_count} → {cleaned_count} rows (dropped {dropped})")

    return df

In [9]:
print("\nCleaning datasets...")
df1 = clean_dataframe(df1, "nl2bash-custom")
df2 = clean_dataframe(df2, "linux-command-dataset")
df3 = clean_dataframe(df3, "unix-commands")


Cleaning datasets...
  nl2bash-custom: 19658 → 11916 rows (dropped 7742)
  linux-command-dataset: 8669 → 7786 rows (dropped 883)
  unix-commands: 100 → 99 rows (dropped 1)


In [10]:
combined = pd.concat([df1, df2, df3], ignore_index=True)
print(f"Combined total: {len(combined)} rows")

Combined total: 19801 rows


In [13]:
before = len(combined)
combined = combined.drop_duplicates(subset=["instruction"])
after = len(combined)
print(f"Dropped {before - after} duplicates")

Dropped 0 duplicates


In [14]:
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)
 
split_index = int(len(combined) * 0.9)
train_df = combined[:split_index]
test_df  = combined[split_index:]
 
print(f"Train set: {len(train_df)} rows")
print(f"Test set:  {len(test_df)} rows")

Train set: 17820 rows
Test set:  1981 rows


In [15]:
train_df.to_json("train.json", orient="records", indent=2)
test_df.to_json("test.json",   orient="records", indent=2)
 
print("Saved train.json")
print("Saved test.json")
 

Saved train.json
Saved test.json
